# 02 - Feature Engineering from Raw Files
This notebook reads canonical raw OHLCV data from data/raw, engineers features, and writes processed model-ready data to data/processed.

In [15]:
from pathlib import Path
import logging
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_settings
from src.features import DEFAULT_FEATURE_COLUMNS, build_model_dataset
from src.logging_utils import configure_logging

configure_logging(logging.INFO)
logger = logging.getLogger("notebook.02_feature_engineering")
logger.info("Notebook initialization complete.")

2026-04-02 17:41:18,520 | INFO | notebook.02_feature_engineering | Notebook initialization complete.


In [16]:
logger.info("Step 1/3: Loading raw OHLCV data from data/raw.")
settings = get_settings()

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_parquet_path = raw_dir / "ohlcv_raw.parquet"
raw_csv_path = raw_dir / "ohlcv_raw.csv"

prices = None
source_path = None
if raw_parquet_path.exists():
    try:
        prices = pd.read_parquet(raw_parquet_path)
        source_path = raw_parquet_path
    except ImportError:
        logger.warning("Parquet engine not available; falling back to CSV raw data.")

if prices is None and raw_csv_path.exists():
    prices = pd.read_csv(raw_csv_path)
    source_path = raw_csv_path

if prices is None:
    raise FileNotFoundError(
        f"Raw data not found. Expected one of: {raw_parquet_path} or {raw_csv_path}. "
        "Run notebook 01 first."
    )

logger.info("Loaded raw rows: %d from %s", len(prices), source_path)
print("Raw source:", source_path)
print('Raw rows:', len(prices))
prices.head()

2026-04-02 17:41:18,535 | INFO | notebook.02_feature_engineering | Step 1/3: Loading raw OHLCV data from data/raw.
2026-04-02 17:41:18,572 | INFO | notebook.02_feature_engineering | Loaded raw rows: 13825 from c:\Users\kenne\OneDrive\Desktop\Quant_Projects_2026\Cross-Sectional-Machine-Learning-Alpha-for-Equities\data\raw\ohlcv_raw.csv


Raw source: c:\Users\kenne\OneDrive\Desktop\Quant_Projects_2026\Cross-Sectional-Machine-Learning-Alpha-for-Equities\data\raw\ohlcv_raw.csv
Raw rows: 13825


,date,ticker,open,high,low,close,volume
0,2015-01-02,AAPL,24.671147,24.682222,23.776350,24.214890,212818400
1,2015-01-02,AMZN,15.629000,15.737500,15.348000,15.426000,55664000
2,2015-01-02,GOOGL,26.411708,26.570398,26.177643,26.260460,26480000
3,2015-01-02,MSFT,39.682632,40.328983,39.580578,39.767677,27913900
4,2015-01-02,NVDA,0.482985,0.486584,0.475307,0.482985,113680000


In [17]:
logger.info("Step 2/3: Engineering features and targets.")
dataset = build_model_dataset(prices, horizon_days=settings.prediction_horizon_days)

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
processed_parquet_path = processed_dir / "model_dataset.parquet"
processed_csv_path = processed_dir / "model_dataset.csv"

parquet_written = False
try:
    dataset.to_parquet(processed_parquet_path, index=False)
    parquet_written = True
except ImportError:
    logger.warning("Parquet engine not available; exporting processed data to CSV only.")

dataset.to_csv(processed_csv_path, index=False)
if parquet_written:
    logger.info("Processed dataset exported to %s and %s", processed_parquet_path, processed_csv_path)
else:
    logger.info("Processed dataset exported to %s", processed_csv_path)

print("Processed csv:", processed_csv_path)
if parquet_written:
    print("Processed parquet:", processed_parquet_path)
print('Dataset rows after feature engineering + cleaning:', len(dataset))
dataset[['date', 'ticker', *DEFAULT_FEATURE_COLUMNS, 'target_fwd_return']].head()

2026-04-02 17:41:18,601 | INFO | notebook.02_feature_engineering | Step 2/3: Engineering features and targets.
2026-04-02 17:41:18,703 | WARNING | notebook.02_feature_engineering | Parquet engine not available; exporting processed data to CSV only.
2026-04-02 17:41:19,047 | INFO | notebook.02_feature_engineering | Processed dataset exported to c:\Users\kenne\OneDrive\Desktop\Quant_Projects_2026\Cross-Sectional-Machine-Learning-Alpha-for-Equities\data\processed\model_dataset.csv


Processed csv: c:\Users\kenne\OneDrive\Desktop\Quant_Projects_2026\Cross-Sectional-Machine-Learning-Alpha-for-Equities\data\processed\model_dataset.csv
Dataset rows after feature engineering + cleaning: 13695


,date,ticker,ret_1d,ret_5d,ret_21d,ma_dist_10,ma_dist_21,vol_21d,rsi_14,vol_chg_5d,target_fwd_return
0,2015-02-03,AAPL,-0.574415,0.347210,0.496487,0.160052,0.237099,-0.250036,0.275913,-0.530969,0.032462
1,2015-02-04,AAPL,0.773084,-0.364095,0.617589,0.237996,0.294397,-0.366567,0.397569,-0.834831,0.048619
2,2015-02-05,AAPL,-0.925189,-0.569806,0.361529,0.025233,0.191345,-0.285942,0.544998,-0.545189,0.054360
3,2015-02-06,AAPL,-1.228270,-0.580964,0.228872,-0.276743,0.094616,-0.264717,0.574685,0.418319,0.068528
4,2015-02-09,AAPL,1.410196,-0.286164,0.167928,-0.179354,0.235564,-0.342958,0.570325,0.088557,0.067742


In [18]:
logger.info("Step 3/3: Sanity-checking per-date cross-sectional sample counts.")
daily_counts = dataset.groupby('date').size().rename('num_names').to_frame()
daily_counts.tail()

2026-04-02 17:41:19,073 | INFO | notebook.02_feature_engineering | Step 3/3: Sanity-checking per-date cross-sectional sample counts.


,num_names
date,
2025-12-16,5
2025-12-17,5
2025-12-18,5
2025-12-19,5
2025-12-22,5
